## Setup the Sample Dataset

In [0]:
# update your destination accordingly
catalog_src = "databricks_airbnb_sample_data"
schema_src = "v01"
volume_src = "sf-listings"
file_src = "sf-airbnb.csv"

# Path configuration
path = f"/Volumes/{catalog_src}/{schema_src}/{volume_src}"

# Check if the file exists:
items = dbutils.fs.ls(path)

file_exist = False
if file not in [item.name for item in items]:
    raise Exception(f"File {file} not found in your volume {path}")
else:
    print(f"File {file} found in your volume {path}")
    file_exist = True

In [0]:
# update your destination accordingly
catalog = "workspace"
schema = "bronze"
table = "sf_airbnb_listings"
table_name = f"{catalog}.{schema}.{table}"

# check if table exists
table_exist = False
if spark.catalog.tableExists(table_name):
    print(f"Table {table_name} exists.")
    table_exist = True
else:
    print(f"Table {table_name} does not exist.")

In [0]:
if file_exist and not table_exist:
    print(f"Reading file {file_src} from volume {path}")
    # Fixed: Add quote and escape options to handle commas in text fields
    df_raw = (spark.read.format("csv")
        .option("header", "true")
        .option("quote", '"')           # Fields wrapped in double quotes
        .option("escape", '"')          # Escaped quotes use double-quote
        .option("multiLine", "true")    # Handle fields spanning multiple lines
        .option("inferSchema", "true")  # Optional: infer data types
        .load(f"{path}/{file_src}"))

    print(f"Writing table {table_name}")
    df_raw.write.format("delta").mode("overwrite").saveAsTable(table_name)

    print(f"Table {table_name} created.")